In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import pandas as pd
from langchain_google_vertexai import ChatVertexAI
from langchain_openai import ChatOpenAI
from linalgo.hub.client import LinalgoClient

from wsd.models import ClusterByMeaningAnnotator, DummyComparator

In [3]:
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
task_id = 'd3ce7764-eb85-4999-b965-c028f539ee33'
task = client.get_task(task_id, verbose=True)

Retrivieving task with id d3ce7764-eb85-4999-b965-c028f539ee33...
Retrieving annotators... (4 found)
Retrieving entities... (7 found)
Retrieving documents... (63 found)
Retrieving annotations... (428 found)


In [ ]:
comparators = [
    DummyComparator(probability=1),
    # ChatOpenAI(temperature=0, model="gpt-4o-mini"),  # Very slow
    # ChatOpenAI(temperature=0, model="gpt-4o"),  # Very slow for some reason
    ChatVertexAI(temperature=0, model="gemini-1.5-flash"),
    ChatVertexAI(temperature=0, model="gemini-1.5-pro"),
    # ChatVertexAI(temperature=0, model="gemini-2.0-flash-exp"),
]
for comparator in comparators:
    model = ClusterByMeaningAnnotator(comparator=comparator)
    y = model.predict(task.documents, verbose=True)
    task.annotators.append(model.annotator)

100%|██████████| 63/63 [03:08<00:00,  2.99s/it]


In [ ]:
from collections import defaultdict 
from itertools import permutations

from linpub.metrics import accuracy


def get_offset(a):
    return int(a.target.selector[0].start_offset)

def get_data():
    """Group annotations by annotator for each documents."""
    data = defaultdict(dict)
    for annotator in task.annotators:
        for doc in task.documents:
            annotations = [a for a in doc.annotations if a.annotator.id == annotator.id]
            annotations = sorted(annotations, key=get_offset)
            data[doc.id][annotator.id] = annotations
    return data

# Compute the cluster accuracy between each annotator pairs.
data = []
for annotator1, annotator2 in permutations(task.annotators[1:], 2):
    y, y_pred = [], []
    for doc, t in get_data().items():
        annotations1 = t[annotator1.id]
        annotations2 = t[annotator2.id]
        if len(annotations1) == len(annotations2):
            y.extend([hash((doc, a.entity.id)) for a in annotations1])
            y_pred.extend([hash((doc, a.entity.id)) for a in annotations2])
    data.append({'y': annotator1.name, 'y_pred': annotator2.name, 'cluster_accuracy': accuracy(y_pred, y)})

In [ ]:
df = pd.DataFrame(data).pivot(index='y', columns='y_pred', values='cluster_accuracy')
df.style.background_gradient(cmap='Greens')

y_pred,Dummy=1,arnaud,gemini-1.5-flash,gemini-1.5-pro,jack,wallie
y,,,,,,
Dummy=1,nan,0.386667,0.347368,0.190909,0.337500,0.300000
arnaud,0.557692,nan,0.784810,0.591398,0.784091,0.806818
gemini-1.5-flash,0.523810,0.826667,nan,0.690909,0.675000,0.737500
gemini-1.5-pro,0.333333,0.733333,0.800000,nan,0.712500,0.762500
jack,0.509434,0.831325,0.675000,0.606383,nan,0.831461
wallie,0.461538,0.876543,0.766234,0.677778,0.860465,nan


In [7]:
df.to_csv('plop.csv', index=False)